# UAV cybersecurity framework

Controlled workflow: OMNeT++ CSV -> validated Send records -> Isolation Forest and Random Forest -> SOC events and Wazuh-oriented export.

The four numerical features are generated by the supplied simulator, not measured radio-network performance. The reference snapshot has 494 events, including 247 Send records (220 Normal and 27 Attack). Historical 795-row outputs are incompatible and remain archived separately.

Run cells from top to bottom in Colab, or use `tools/run_analysis.py` from the project root to execute this same notebook locally. Local execution defaults to the AI/SOC analysis only. Set `UAV_RUN_PQC=1` in a prepared environment to enable the optional representative-payload PQC demonstration; its Colab installation cells require a disposable Linux runtime. PQC results from other runs are never attached to the current events.


In [22]:
# Install dependencies in Colab only; local dependencies come from requirements-analysis.txt.
import os
import importlib.util
try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    get_ipython().run_line_magic("pip", "install -q pandas numpy scikit-learn matplotlib seaborn cryptography")


In [23]:
# CELL 2 — IMPORT NORMAL LIBRARIES
import os
import sys
import json
import time
import hashlib
import logging
import shutil
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    auc
)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from cryptography.hazmat.primitives.ciphers.aead import AESGCM
if IN_COLAB:
    from google.colab import files
from pathlib import Path

RUN_PQC = os.environ.get("UAV_RUN_PQC", "1" if IN_COLAB else "0") == "1"
RUN_ID = os.environ.get("UAV_RUN_ID", datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%S%fZ"))
OUTPUT_DIR = os.environ.get("UAV_OUTPUT_DIR", os.path.join("outputs", RUN_ID))
if os.path.isdir(OUTPUT_DIR) and os.listdir(OUTPUT_DIR):
    raise FileExistsError("Output directory must be empty to prevent mixing runs: " + OUTPUT_DIR)
os.makedirs(OUTPUT_DIR, exist_ok=True)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

logging.basicConfig(
    filename=os.path.join(OUTPUT_DIR, "soc_alerts.log"),
    level=logging.INFO,
    force=True,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

print("Normal AI/SOC libraries loaded successfully.")
print("Python:", sys.version.split()[0])


Normal AI/SOC libraries loaded successfully.
Python: 3.12.3


In [ ]:
# CELL 3 — UPLOAD AND VALIDATE OMNeT++ CSV
uploaded_file_name = os.environ.get("UAV_INPUT_CSV")
if not uploaded_file_name:
    if IN_COLAB:
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        uploaded_file_name = next(iter(uploaded.keys()))
    else:
        # VS Code can start the kernel in the notebook folder or the project root.
        relative_csv = Path("01_OMNeT_Simulation/UAV_SOC_Project/simulations/uav_traffic.csv")
        working_directory = Path.cwd().resolve()
        candidates = [base / relative_csv for base in (working_directory, *working_directory.parents)]
        source_path = next((candidate for candidate in candidates if candidate.is_file()), None)
        if source_path is None:
            raise FileNotFoundError(
                "Cannot find the simulation CSV from " + str(working_directory)
                + ". Open this notebook inside the project folder or set UAV_INPUT_CSV "
                "to the full path of uav_traffic.csv. Searched: "
                + ", ".join(str(candidate) for candidate in candidates)
            )
        uploaded_file_name = str(source_path)

# Preserve explicit environment overrides and Colab uploads; validate before reading.
source_path = Path(uploaded_file_name).expanduser().resolve()
if not source_path.is_file():
    raise FileNotFoundError(
        "Input CSV does not exist: " + str(source_path)
        + ". Check UAV_INPUT_CSV or upload the CSV again in Colab."
    )
uploaded_file_name = str(source_path)
source_sha256 = hashlib.sha256(source_path.read_bytes()).hexdigest()

if not uploaded_file_name.lower().endswith(".csv"):
    raise ValueError("Please upload a genuine CSV file.")

with open(uploaded_file_name, "rb") as f:
    first_bytes = f.read(8)

if first_bytes.startswith(b"PK"):
    raise ValueError(
        "This looks like a Numbers/ZIP file renamed to .csv. "
        "Export it properly as CSV first."
    )

REQUIRED_OMNET_COLUMNS = [
    "Time", "UAV", "Event", "PacketSize", "Source",
    "Destination", "Latency", "PacketLoss", "Throughput", "Label"
]

df = pd.read_csv(uploaded_file_name)
df.columns = df.columns.astype(str).str.strip()

missing_columns = [
    column for column in REQUIRED_OMNET_COLUMNS
    if column not in df.columns
]
unexpected_columns = [
    column for column in df.columns
    if column not in REQUIRED_OMNET_COLUMNS
]

if list(df.columns) != REQUIRED_OMNET_COLUMNS:
    raise ValueError(
        "Invalid OMNeT++ CSV schema. The CSV must contain exactly these "
        f"columns in this order: {REQUIRED_OMNET_COLUMNS}. "
        f"Missing columns: {missing_columns}. "
        f"Unexpected columns: {unexpected_columns}. "
        f"Detected columns: {list(df.columns)}"
    )

raw_csv_rows = int(len(df))

print("Loaded:", uploaded_file_name)
print("Raw CSV shape:", df.shape)
print("Validated OMNeT++ columns:", list(df.columns))
display(df.head())

In [ ]:
# CELL 4 — FILTER SEND EVENTS AND PREPARE SIMULATOR-GENERATED FIELDS
# The controlled experiment models transmitted UAV records only.
df["source_record_id"] = df.index.astype(int)

df["Event"] = df["Event"].astype(str).str.strip()
df = df[df["Event"].str.casefold() == "send"].copy()

if df.empty:
    raise ValueError("No Event == 'Send' rows were found in the OMNeT++ CSV.")

df["Label"] = df["Label"].astype(str).str.strip()
valid_labels = {"Normal", "Attack"}
unknown_labels = sorted(set(df["Label"].dropna()) - valid_labels)

if unknown_labels or df["Label"].isna().any():
    raise ValueError(
        "OMNeT++ Label must contain only 'Normal' or 'Attack'. "
        f"Unknown values: {unknown_labels}"
    )

df["label"] = df["Label"].map({"Normal": 0, "Attack": 1}).astype(int)

real_send_rows = int(len(df))
real_normal_rows = int((df["label"] == 0).sum())
real_attack_rows = int((df["label"] == 1).sum())

if real_attack_rows == 0:
    raise ValueError(
        "The filtered OMNeT++ Send-event dataset contains zero Attack rows. "
        "Synthetic attack rows will not be generated."
    )

EXPECTED_SEND_ROWS = 247
EXPECTED_NORMAL_ROWS = 220
EXPECTED_ATTACK_ROWS = 27

if (
    real_send_rows != EXPECTED_SEND_ROWS
    or real_normal_rows != EXPECTED_NORMAL_ROWS
    or real_attack_rows != EXPECTED_ATTACK_ROWS
):
    raise ValueError(
        "Unexpected controlled OMNeT++ dataset counts. "
        f"Expected Send/Normal/Attack = "
        f"{EXPECTED_SEND_ROWS}/{EXPECTED_NORMAL_ROWS}/{EXPECTED_ATTACK_ROWS}, "
        f"but found {real_send_rows}/{real_normal_rows}/{real_attack_rows}."
    )

# Preserve the source UAV identity. uav_id is identification metadata only.
df["uav_name"] = df["UAV"].astype(str).str.strip()
uav_encoder = LabelEncoder()
df["uav_id"] = uav_encoder.fit_transform(df["uav_name"]) + 1
uav_mapping = {
    name: int(index + 1)
    for index, name in enumerate(uav_encoder.classes_)
}

rename_map = {
    "Time": "simulation_time",
    "PacketSize": "packet_size",
    "Latency": "avg_latency_ms",
    "PacketLoss": "packet_loss_pct",
    "Throughput": "throughput"
}
df = df.rename(columns=rename_map)

REAL_MODEL_FEATURES = [
    "packet_size",
    "avg_latency_ms",
    "packet_loss_pct",
    "throughput"
]

for column in ["simulation_time"] + REAL_MODEL_FEATURES:
    df[column] = pd.to_numeric(df[column], errors="coerce")

invalid_required_rows = df[["simulation_time"] + REAL_MODEL_FEATURES].isna().any(axis=1)
if invalid_required_rows.any():
    raise ValueError(
        f"{int(invalid_required_rows.sum())} Send rows contain invalid required "
        "OMNeT++ Time, PacketSize, Latency, PacketLoss, or Throughput values."
    )

features = REAL_MODEL_FEATURES.copy()
synthetic_attack_rows_added = 0

df["attack_type"] = np.where(
    df["label"] == 1,
    "OMNeT++ abnormal communication behaviour",
    "Normal"
)

print("Controlled OMNeT++ Send-event dataset validated.")
print("Send rows:", real_send_rows)
print("Normal rows:", real_normal_rows)
print("Attack rows:", real_attack_rows)
print("Synthetic attack rows added:", synthetic_attack_rows_added)
print("UAV encoding (identification/export only):", uav_mapping)
print("Real model features:", features)

In [ ]:
# CELL 5 — BUILD REAL OMNeT++ MODELLING DATASET
# No synthetic attack rows or randomly generated model features are added.
dataset_columns = [
    "source_record_id",
    "simulation_time",
    "uav_name",
    "uav_id",
    "packet_size",
    "avg_latency_ms",
    "packet_loss_pct",
    "throughput",
    "label",
    "attack_type"
]

dataset = df[dataset_columns].copy()
dataset = dataset.replace([np.inf, -np.inf], np.nan)

if dataset.isna().any().any():
    raise ValueError("Prepared dataset contains missing or infinite values.")

if len(dataset) != real_send_rows:
    raise RuntimeError("Prepared dataset row count changed unexpectedly.")

dataset.to_csv(
    os.path.join(OUTPUT_DIR, "omnet_uav_ai_dataset.csv"),
    index=False
)

print("Final controlled OMNeT++ dataset shape:", dataset.shape)
print(dataset["label"].value_counts().sort_index())
display(dataset.head())

In [ ]:
# CELL 6 — STRATIFIED 70/30 TRAIN / TEST SPLIT
if dataset["label"].nunique() < 2:
    raise ValueError("Need both Normal and Attack classes.")

train_df, test_df = train_test_split(
    dataset,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=dataset["label"]
)

y_true = test_df["label"].astype(int).values

iso_scaler = StandardScaler()
rf_scaler = StandardScaler()

normal_train = train_df[train_df["label"] == 0]

X_train_iso = iso_scaler.fit_transform(normal_train[features])
X_test_iso = iso_scaler.transform(test_df[features])

X_train_rf = rf_scaler.fit_transform(train_df[features])
X_test_rf = rf_scaler.transform(test_df[features])
y_train_rf = train_df["label"].astype(int).values

# This uses training labels only and does not inspect test labels.
observed_training_attack_proportion = float(train_df["label"].mean())
iso_contamination = float(np.clip(
    observed_training_attack_proportion,
    0.01,
    0.50
))

print("Controlled OMNeT++ simulation split")
print("Train rows:", len(train_df), "Test rows:", len(test_df))
print("Training Normal rows:", int((train_df["label"] == 0).sum()))
print("Training Attack rows:", int((train_df["label"] == 1).sum()))
print(
    "Observed training attack proportion:",
    f"{observed_training_attack_proportion:.6f}"
)
print(
    "Selected Isolation Forest contamination:",
    f"{iso_contamination:.6f}",
    "(derived from the controlled training split only)"
)
# Source CSV data-row IDs (zero-based, excluding header) make the split auditable.
split_membership = pd.concat([
    train_df[["source_record_id"]].assign(split="train"),
    test_df[["source_record_id"]].assign(split="test")
]).sort_values("source_record_id")
split_membership.to_csv(os.path.join(OUTPUT_DIR, "split_membership.csv"), index=False)


In [ ]:
# CELL 7 — ISOLATION FOREST ON NORMAL TRAINING SAMPLES
iso_model = IsolationForest(
    n_estimators=200,
    contamination=iso_contamination,
    random_state=RANDOM_STATE
)

iso_model.fit(X_train_iso)

iso_raw = iso_model.predict(X_test_iso)
iso_pred = np.where(iso_raw == -1, 1, 0)
iso_scores = -iso_model.decision_function(X_test_iso)

print("Isolation Forest complete using only real OMNeT++ model features.")

In [ ]:
# CELL 8 — RANDOM FOREST
rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf_model.fit(X_train_rf, y_train_rf)

rf_pred = rf_model.predict(X_test_rf)
rf_scores = rf_model.predict_proba(X_test_rf)[:, 1]

print("Random Forest complete.")


In [ ]:
# CELL 9 — CONTROLLED OMNeT++ SIMULATION EVALUATION
from sklearn.metrics import roc_auc_score

def metrics_row(name, y_true, y_pred, y_score):
    return {
        "Model": name,
        "Evaluation Scope": "Controlled OMNeT++ simulation results",
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1 Score": f1_score(y_true, y_pred, zero_division=0),
        "ROC-AUC": roc_auc_score(y_true, y_score)
    }

comparison_df = pd.DataFrame([
    metrics_row("Isolation Forest", y_true, iso_pred, iso_scores),
    metrics_row("Random Forest", y_true, rf_pred, rf_scores)
])

comparison_df.to_csv(
    os.path.join(OUTPUT_DIR, "model_comparison.csv"),
    index=False
)

report_header = (
    "Controlled OMNeT++ simulation results\n"
    "These results are not real-world UAV performance.\n\n"
)

with open(os.path.join(OUTPUT_DIR, "classification_report.txt"), "w") as f:
    f.write(report_header)
    f.write("Isolation Forest Report\n")
    f.write(classification_report(y_true, iso_pred, digits=4))
    f.write("\n\nRandom Forest Report\n")
    f.write(classification_report(y_true, rf_pred, digits=4))

display(comparison_df)

In [ ]:
# CELL 10 — HONEST SOC EVENT GENERATION
class SOCSimulator:
    def __init__(self):
        self.alerts = []

    @staticmethod
    def classify_event(row):
        if row["packet_loss_pct"] > 10 or row["avg_latency_ms"] > 100:
            return "Possible Communication Degradation / Jamming-like Behaviour"
        return "UAV Network Anomaly"

    @staticmethod
    def classify_severity(row, iso_prediction, rf_prediction):
        severe_measurements = (
            row["packet_loss_pct"] > 20
            or row["avg_latency_ms"] > 140
        )
        both_models = int(iso_prediction) == 1 and int(rf_prediction) == 1
        if severe_measurements and both_models:
            return "High"
        return "Medium"

    def process_event(self, row):
        iso_prediction = int(row["isolation_forest_prediction"])
        rf_prediction = int(row["random_forest_prediction"])

        if iso_prediction == 1 or rf_prediction == 1:
            alert = {
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "run_id": RUN_ID,
                "source_record_id": int(row["source_record_id"]),
                "event_source": "UAV_Controlled_OMNeT_AI_SOC",
                "evaluation_scope": "Controlled OMNeT++ simulation results",
                "uav_name": str(row["uav_name"]),
                "uav_id": int(row["uav_id"]),
                "simulation_time": float(row["simulation_time"]),
                "event_type": self.classify_event(row),
                "severity": self.classify_severity(
                    row, iso_prediction, rf_prediction
                ),
                "isolation_forest_prediction": iso_prediction,
                "random_forest_prediction": rf_prediction,
                "anomaly_score": float(row["anomaly_score"]),
                "random_forest_attack_probability": float(
                    row["random_forest_attack_probability"]
                ),
                "packet_size": float(row["packet_size"]),
                "avg_latency_ms": float(row["avg_latency_ms"]),
                "packet_loss_pct": float(row["packet_loss_pct"]),
                "throughput": float(row["throughput"]),
                "ground_truth_label": int(row["label"])
            }
            self.alerts.append(alert)
            logging.warning(json.dumps(alert))

soc = SOCSimulator()

test_output = test_df.copy()
test_output["isolation_forest_prediction"] = iso_pred
test_output["random_forest_prediction"] = rf_pred
test_output["anomaly_score"] = iso_scores
test_output["random_forest_attack_probability"] = rf_scores

for _, row in test_output.iterrows():
    soc.process_event(row)

soc_alerts_df = pd.DataFrame(soc.alerts)
soc_alerts_df.to_csv(
    os.path.join(OUTPUT_DIR, "soc_alerts.csv"),
    index=False
)

test_output.to_csv(
    os.path.join(OUTPUT_DIR, "uav_test_predictions.csv"),
    index=False
)

print("SOC alerts generated:", len(soc.alerts))
print(
    "SOC scope: communication degradation/jamming-like behaviour and "
    "generic UAV network anomalies only."
)

In [ ]:
# CELL 11 — CONTROLLED OMNeT++ SIMULATION GRAPHS
cm_iso = confusion_matrix(y_true, iso_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm_iso, annot=True, fmt="d",
            xticklabels=["Normal","Attack"],
            yticklabels=["Normal","Attack"])
plt.title("Controlled OMNeT++ Results - Isolation Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"confusion_matrix_isolation_forest.png"), dpi=300)
plt.show()

cm_rf = confusion_matrix(y_true, rf_pred)
plt.figure(figsize=(6,4))
sns.heatmap(cm_rf, annot=True, fmt="d",
            xticklabels=["Normal","Attack"],
            yticklabels=["Normal","Attack"])
plt.title("Controlled OMNeT++ Results - Random Forest")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"confusion_matrix_random_forest.png"), dpi=300)
plt.show()

fpr_iso, tpr_iso, _ = roc_curve(y_true, iso_scores)
roc_auc_iso = auc(fpr_iso, tpr_iso)

fpr_rf, tpr_rf, _ = roc_curve(y_true, rf_scores)
roc_auc_rf = auc(fpr_rf, tpr_rf)

plt.figure(figsize=(6,5))
plt.plot(fpr_iso, tpr_iso, label=f"Isolation Forest AUC = {roc_auc_iso:.3f}")
plt.plot(fpr_rf, tpr_rf, label=f"Random Forest AUC = {roc_auc_rf:.3f}")
plt.plot([0,1],[0,1],"--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Controlled OMNeT++ Simulation ROC Comparison")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,"roc_curve_comparison.png"), dpi=300)
plt.show()

print("Controlled OMNeT++ AI/SOC evaluation complete.")
print("These plots do not represent real-world UAV performance.")

# PQC Section — Representative Payload Cryptographic Proof-of-Concept

The following section benchmarks ML-KEM-768 with AES-256-GCM using one representative AI/SOC payload. It does **not** imply that every OMNeT++ packet or SOC event was cryptographically protected.

In [ ]:
if RUN_PQC and IN_COLAB:
    # ============================================================
    # CELL 12 — INSTALL LIBOQS MANUALLY
    # ============================================================

    !apt-get update -qq
    !apt-get install -y -qq git cmake ninja-build build-essential libssl-dev

    # Remove previous incomplete installations
    !rm -rf /content/liboqs
    !rm -rf /content/liboqs-python
    !rm -rf /root/_oqs

    # Download liboqs
    !git clone --depth=1 https://github.com/open-quantum-safe/liboqs.git /content/liboqs

    # Configure liboqs
    !cmake \
        -S /content/liboqs \
        -B /content/liboqs/build \
        -GNinja \
        -DBUILD_SHARED_LIBS=ON \
        -DOQS_BUILD_ONLY_LIB=ON \
        -DCMAKE_INSTALL_PREFIX=/usr/local

    # Build
    !cmake --build /content/liboqs/build --parallel 2

    # Install
    !cmake --install /content/liboqs/build

    # Refresh shared library cache
    !ldconfig

    # Check installation
    !find /usr/local/lib -name "liboqs*" -print

    print("✅ liboqs installation completed")

else:
    print("Colab liboqs installation skipped.")


In [ ]:
if RUN_PQC:
    if IN_COLAB:
        # ============================================================
        # CELL 13 — INSTALL LIBOQS-PYTHON AND TEST ML-KEM
        # ============================================================

        import os

        # Download Python wrapper
        !git clone --depth=1 https://github.com/open-quantum-safe/liboqs-python.git /content/liboqs-python

        # Install Python wrapper
        !pip install -q /content/liboqs-python

        # Tell Python where liboqs is installed
        os.environ["OQS_INSTALL_PATH"] = "/usr/local"


    import oqs

    print("liboqs version:", oqs.oqs_version())
    print("liboqs-python version:", oqs.oqs_python_version())

    enabled_kems = oqs.get_enabled_kem_mechanisms()

    print("\nAvailable ML-KEM algorithms:")
    print([x for x in enabled_kems if "ML-KEM" in x])

    PQC_ALGORITHM = "ML-KEM-768"

    if PQC_ALGORITHM in enabled_kems:
        print("\n✅ ML-KEM-768 is READY")
    else:
        raise RuntimeError("❌ ML-KEM-768 is not available")

else:
    print("PQC disabled for this run.")


In [ ]:
if RUN_PQC:
    # CELL 14 — REPRESENTATIVE UAV PAYLOAD CRYPTOGRAPHIC PROOF-OF-CONCEPT
    attack_rows = test_output[test_output["label"] == 1]
    representative = attack_rows.iloc[0] if len(attack_rows) else test_output.iloc[0]
    representative_row_index = representative.name

    uav_payload = {
        "scope": "Representative payload cryptographic proof-of-concept",
        "uav_name": str(representative["uav_name"]),
        "uav_id": int(representative["uav_id"]),
        "simulation_time": float(representative["simulation_time"]),
        "packet_size": float(representative["packet_size"]),
        "avg_latency_ms": float(representative["avg_latency_ms"]),
        "packet_loss_pct": float(representative["packet_loss_pct"]),
        "throughput": float(representative["throughput"]),
        "isolation_forest_prediction": int(
            representative["isolation_forest_prediction"]
        ),
        "random_forest_prediction": int(
            representative["random_forest_prediction"]
        ),
        "command": "RETURN_TO_BASE"
    }

    uav_message = json.dumps(uav_payload, separators=(",", ":")).encode("utf-8")

    print("Representative payload cryptographic proof-of-concept")
    print(uav_message.decode())
    print("Plaintext bytes:", len(uav_message))
else:
    print("Optional PQC step skipped.")


In [ ]:
if RUN_PQC:
    # CELL 15 — REPRESENTATIVE ML-KEM-768 + AES-256-GCM PROOF-OF-CONCEPT
    with oqs.KeyEncapsulation(PQC_ALGORITHM) as gcs:
        t0 = time.perf_counter_ns()
        public_key = gcs.generate_keypair()
        keygen_ms = (time.perf_counter_ns() - t0) / 1_000_000

        with oqs.KeyEncapsulation(PQC_ALGORITHM) as sender:
            t0 = time.perf_counter_ns()
            kem_ciphertext, shared_secret_sender = sender.encap_secret(public_key)
            encapsulation_ms = (time.perf_counter_ns() - t0) / 1_000_000

        t0 = time.perf_counter_ns()
        shared_secret_receiver = gcs.decap_secret(kem_ciphertext)
        decapsulation_ms = (time.perf_counter_ns() - t0) / 1_000_000

    shared_secret_match = shared_secret_sender == shared_secret_receiver

    aes_key = hashlib.sha256(shared_secret_sender).digest()
    aesgcm = AESGCM(aes_key)
    nonce = os.urandom(12)

    t0 = time.perf_counter_ns()
    encrypted_message = aesgcm.encrypt(nonce, uav_message, None)
    aes_encrypt_ms = (time.perf_counter_ns() - t0) / 1_000_000

    t0 = time.perf_counter_ns()
    decrypted_message = aesgcm.decrypt(nonce, encrypted_message, None)
    aes_decrypt_ms = (time.perf_counter_ns() - t0) / 1_000_000

    representative_payload_verified = decrypted_message == uav_message

    print("Shared secret match:", shared_secret_match)
    print("Representative payload verified:", representative_payload_verified)
    print("Public key bytes:", len(public_key))
    print("KEM ciphertext bytes:", len(kem_ciphertext))
    print("Shared secret bytes:", len(shared_secret_sender))
    print(f"Key generation: {keygen_ms:.4f} ms")
    print(f"Encapsulation: {encapsulation_ms:.4f} ms")
    print(f"Decapsulation: {decapsulation_ms:.4f} ms")
    print(f"AES encryption: {aes_encrypt_ms:.4f} ms")
    print(f"AES decryption: {aes_decrypt_ms:.4f} ms")
else:
    print("Optional PQC step skipped.")


In [ ]:
if RUN_PQC:
    # CELL 16 — PQC BENCHMARK
    PQC_ITERATIONS = 200
    results = []

    for i in range(PQC_ITERATIONS):
        gcs = oqs.KeyEncapsulation(PQC_ALGORITHM)

        t0 = time.perf_counter_ns()
        public_key_i = gcs.generate_keypair()
        keygen_i = (time.perf_counter_ns() - t0) / 1_000_000

        sender = oqs.KeyEncapsulation(PQC_ALGORITHM)

        t0 = time.perf_counter_ns()
        ct_i, ss_sender = sender.encap_secret(public_key_i)
        encaps_i = (time.perf_counter_ns() - t0) / 1_000_000

        t0 = time.perf_counter_ns()
        ss_receiver = gcs.decap_secret(ct_i)
        decaps_i = (time.perf_counter_ns() - t0) / 1_000_000

        aes_key_i = hashlib.sha256(ss_sender).digest()
        aes_i = AESGCM(aes_key_i)
        nonce_i = os.urandom(12)

        t0 = time.perf_counter_ns()
        encrypted_i = aes_i.encrypt(nonce_i, uav_message, None)
        aes_enc_i = (time.perf_counter_ns() - t0) / 1_000_000

        t0 = time.perf_counter_ns()
        decrypted_i = aes_i.decrypt(nonce_i, encrypted_i, None)
        aes_dec_i = (time.perf_counter_ns() - t0) / 1_000_000

        success_i = (
            ss_sender == ss_receiver
            and decrypted_i == uav_message
        )

        results.append({
            "iteration": i + 1,
            "algorithm": PQC_ALGORITHM,
            "keygen_ms": keygen_i,
            "encapsulation_ms": encaps_i,
            "decapsulation_ms": decaps_i,
            "aes_encrypt_ms": aes_enc_i,
            "aes_decrypt_ms": aes_dec_i,
            "public_key_bytes": len(public_key_i),
            "kem_ciphertext_bytes": len(ct_i),
            "shared_secret_bytes": len(ss_sender),
            "plaintext_bytes": len(uav_message),
            "encrypted_message_bytes": len(encrypted_i),
            "success": success_i
        })

        sender.free()
        gcs.free()

    pqc_results_df = pd.DataFrame(results)

    pqc_results_df.to_csv(
        os.path.join(OUTPUT_DIR, "pqc_benchmark_results.csv"),
        index=False
    )

    pqc_summary = pqc_results_df[
        [
            "keygen_ms",
            "encapsulation_ms",
            "decapsulation_ms",
            "aes_encrypt_ms",
            "aes_decrypt_ms"
        ]
    ].agg(["mean", "median", "std", "min", "max"])

    pqc_summary.to_csv(
        os.path.join(OUTPUT_DIR, "pqc_performance_summary.csv")
    )

    display(pqc_summary)
    print("PQC success rate:", pqc_results_df["success"].mean() * 100, "%")

else:
    print("Optional PQC step skipped.")


In [ ]:
if RUN_PQC:
    # CELL 17 — PQC GRAPHS
    means = pqc_results_df[
        [
            "keygen_ms",
            "encapsulation_ms",
            "decapsulation_ms",
            "aes_encrypt_ms",
            "aes_decrypt_ms"
        ]
    ].mean()

    plt.figure(figsize=(10,5))
    plt.bar(
        ["Keygen", "Encapsulation", "Decapsulation", "AES Encrypt", "AES Decrypt"],
        means.values
    )
    plt.ylabel("Average Time (ms)")
    plt.title("ML-KEM-768 + AES-256-GCM UAV Security Overhead")
    plt.xticks(rotation=15)
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "pqc_performance.png"),
        dpi=300
    )
    plt.show()

    plt.figure(figsize=(9,5))
    plt.plot(
        pqc_results_df["iteration"],
        pqc_results_df["encapsulation_ms"],
        label="Encapsulation"
    )
    plt.plot(
        pqc_results_df["iteration"],
        pqc_results_df["decapsulation_ms"],
        label="Decapsulation"
    )
    plt.xlabel("Iteration")
    plt.ylabel("Time (ms)")
    plt.title("ML-KEM-768 Latency Across Benchmark Iterations")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(
        os.path.join(OUTPUT_DIR, "mlkem_latency.png"),
        dpi=300
    )
    plt.show()

else:
    print("Optional PQC step skipped.")


In [ ]:
# Export this run's predictions and alert-only NDJSON, with explicit PQC status.
pqc_metadata = {
    "pqc_status": "not_run",
    "pqc_scope": "PQC not executed for this run; individual event not cryptographically protected",
    "pqc_representative_demo_verified": None
}
if RUN_PQC:
    pqc_metadata.update({
        "pqc_status": "completed",
        "pqc_algorithm_benchmarked": PQC_ALGORITHM,
        "pqc_scope": "Representative payload proof-of-concept; individual event not cryptographically protected",
        "pqc_representative_demo_verified": bool(representative_payload_verified),
        "pqc_benchmark_mean_encapsulation_ms": float(pqc_results_df["encapsulation_ms"].mean()),
        "pqc_benchmark_mean_decapsulation_ms": float(pqc_results_df["decapsulation_ms"].mean())
    })
wazuh_output = test_output.copy()
wazuh_output["run_id"] = RUN_ID
wazuh_output["evaluation_scope"] = "Controlled OMNeT++ simulation results"
for key, value in pqc_metadata.items():
    wazuh_output[key] = value
wazuh_csv_path = os.path.join(OUTPUT_DIR, "uav_soc_pqc_wazuh_output.csv")
wazuh_output.to_csv(wazuh_csv_path, index=False)
wazuh_event_path = os.path.join(OUTPUT_DIR, "uav_soc_events.json")
with open(wazuh_event_path, "w", encoding="utf-8") as f:
    for alert in soc.alerts:
        f.write(json.dumps(dict(alert, **pqc_metadata)) + "\n")
print("Wazuh-oriented CSV:", wazuh_csv_path)
print("Alert-only NDJSON records:", len(soc.alerts))
display(wazuh_output.head())


In [ ]:
# CELL 19 — FINAL FRAMEWORK SUMMARY WITH PROVENANCE
framework_summary = {
    "evaluation_scope": "Controlled OMNeT++ simulation results",
    "schema_version": "2.0",
    "run_id": RUN_ID,
    "source_sha256": source_sha256,
    "feature_origin": "Simulator-generated values; not measured radio-network performance",
    "random_state": RANDOM_STATE,
    "real_world_performance_claim": False,
    "source_csv": Path(uploaded_file_name).name,
    "raw_csv_rows": raw_csv_rows,
    "modelled_event": "Send",
    "real_send_rows": real_send_rows,
    "real_normal_rows": real_normal_rows,
    "real_attack_rows": real_attack_rows,
    "synthetic_attack_rows_added": synthetic_attack_rows_added,
    "real_model_features": REAL_MODEL_FEATURES,
    "uav_id_usage": "Identification and export only; not an ML feature",
    "dataset_rows": int(len(dataset)),
    "training_rows": int(len(train_df)),
    "test_rows": int(len(test_output)),
    "training_attack_proportion": observed_training_attack_proportion,
    "isolation_forest_contamination": iso_contamination,
    "soc_alerts_generated": int(len(soc.alerts)),
    "soc_scope": (
        "Possible communication degradation/jamming-like behaviour and "
        "generic UAV network anomalies based on controlled OMNeT++ "
        "latency, packet-loss, packet-size and throughput measurements"
    ),
    "isolation_forest": {
        "accuracy": float(accuracy_score(y_true, iso_pred)),
        "precision": float(precision_score(y_true, iso_pred, zero_division=0)),
        "recall": float(recall_score(y_true, iso_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, iso_pred, zero_division=0)),
        "roc_auc": float(roc_auc_iso)
    },
    "random_forest": {
        "accuracy": float(accuracy_score(y_true, rf_pred)),
        "precision": float(precision_score(y_true, rf_pred, zero_division=0)),
        "recall": float(recall_score(y_true, rf_pred, zero_division=0)),
        "f1_score": float(f1_score(y_true, rf_pred, zero_division=0)),
        "roc_auc": float(roc_auc_rf)
    },
    "pqc": {
        "status": "completed",
        "scope": "Representative payload cryptographic proof-of-concept",
        "individual_events_encrypted": False,
        "representative_demo_verified": bool(
            representative_payload_verified
        ),
        "algorithm": PQC_ALGORITHM,
        "symmetric_cipher": "AES-256-GCM",
        "iterations": PQC_ITERATIONS,
        "success_rate_pct": float(pqc_results_df["success"].mean() * 100),
        "mean_keygen_ms": float(pqc_results_df["keygen_ms"].mean()),
        "mean_encapsulation_ms": float(
            pqc_results_df["encapsulation_ms"].mean()
        ),
        "mean_decapsulation_ms": float(
            pqc_results_df["decapsulation_ms"].mean()
        ),
        "mean_aes_encrypt_ms": float(
            pqc_results_df["aes_encrypt_ms"].mean()
        ),
        "mean_aes_decrypt_ms": float(
            pqc_results_df["aes_decrypt_ms"].mean()
        )
    } if RUN_PQC else {
        "status": "not_run",
        "scope": "AI/SOC-only refresh; no PQC benchmarks attached",
        "individual_events_encrypted": False,
        "representative_demo_verified": None
    }
}

with open(
    os.path.join(OUTPUT_DIR, "framework_summary.json"),
    "w"
) as f:
    json.dump(framework_summary, f, indent=2)

print(json.dumps(framework_summary, indent=2))

In [ ]:
# Colab downloads a run-specific archive. Local runner preserves the executed notebook.
logging.shutdown()
print("Generated files:", sorted(os.listdir(OUTPUT_DIR)))
if IN_COLAB:
    zip_path = shutil.make_archive(OUTPUT_DIR + "_bundle", "zip", OUTPUT_DIR)
    files.download(zip_path)
else:
    print("Local output directory:", OUTPUT_DIR)
